##### Copyright 2025 Perceptron AI.

In [ ]:
# Licensed under the MIT License (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://opensource.org/licenses/MIT
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Capability — Image Q&A (Perceptron Mk1)
Ask natural-language questions about a scene and receive answers plus bounding boxes for supporting evidence.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/perceptron-ai-inc/perceptron/blob/main/cookbook/recipes/capabilities/perceptron-mk1/image-qa.ipynb)

## Install dependencies
Install the SDK and Pillow for inline previews.

In [ ]:
%pip install --upgrade perceptron --quiet

## Download assets

In [ ]:
IMAGE_URL = "https://raw.githubusercontent.com/perceptron-ai-inc/perceptron/main/cookbook/_shared/assets/capabilities/qna/studio_scene.webp"

!curl -so studio_scene.webp {IMAGE_URL}

## Configure the Perceptron client
Authenticate once and point the SDK at Perceptron Mk1.

In [ ]:
import os
from pathlib import Path

from IPython.display import display
from PIL import Image, ImageDraw

from perceptron import configure, image, question

api_key = os.getenv("PERCEPTRON_API_KEY", "<your Perceptron API key>")
if not api_key or api_key.startswith("<"):
    raise RuntimeError("Set PERCEPTRON_API_KEY or replace the placeholder in this cell.")

configure(
    provider="perceptron",
    model="perceptron-mk1",
    api_key=api_key,
)

SCENE_PATH = "studio_scene.webp"
ANNOTATED_PATH = Path("studio_scene_annotated.png")

## Ask a grounded question
Request boxed evidence so the model cites the regions that justify its answer.

In [ ]:
display(Image.open(SCENE_PATH))
QUESTION = "What is the single focal point of this scene? Pick just one and box only that element."
qa_result = question(image(str(SCENE_PATH)), QUESTION, expects="box")
print(qa_result.text)
boxes = qa_result.boxes or []
print(f"Returned {len(boxes)} supporting regions")

## Visualize supporting evidence
Convert the normalized boxes to pixels and render the overlay.

In [ ]:
img = Image.open(SCENE_PATH).convert("RGB")
draw = ImageDraw.Draw(img)

if boxes:

    def to_px(point):
        return point.x / 1000 * img.width, point.y / 1000 * img.height

    for box in boxes:
        top_left = to_px(box.top_left)
        bottom_right = to_px(box.bottom_right)
        draw.rectangle([top_left, bottom_right], outline="crimson", width=3)
        label = box.mention or "reference"
        draw.text((top_left[0], max(top_left[1] - 18, 0)), label, fill="crimson")
else:
    print("No regions returned; adjust the question or expects parameter.")

img.save(ANNOTATED_PATH)
display(img)
print(f"Saved grounded answer to {ANNOTATED_PATH}")

## Conclusion & next steps
- Tune `QUESTION` toward inspections, instructions, or policy compliance checks.
- Swap `expects` to `"text"` for free-form answers or keep `"box"` / `"point"` for grounded citations.
- Pass `reasoning=True` when the answer benefits from deeper analysis.
- For video Q&A, see the [Video Q&A](https://github.com/perceptron-ai-inc/perceptron/blob/main/cookbook/recipes/capabilities/perceptron-mk1/video-qa.ipynb) notebook.